# Verify STMUS

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAW_DATA_DIR = Path("/scratch/data/stmus_raw_20260523/Polito-Radboud-DeepLearningUS/")

In [ ]:
# Create a dataframe for the images. File path format:
# (BB|GM|TA)/(Healthy|Pathological)/(Images|Masks)/anon_{subject_id:4d}_{frame_id:d}.png


def parse_image_path(path):
    parts = path.parts
    return {
        "subject_id": int(parts[-1].split("_")[1]),
        "frame_id": int(parts[-1].split("_")[2].split(".")[0]),
        "echogenicity": parts[-3],
        "muscle": parts[-4],
        "image_path": path,
        "mask_path": path.parent.parent / "Masks" / parts[-1],
    }


image_paths = sorted(RAW_DATA_DIR.glob("**/Images/*.png"))
df = pd.DataFrame([parse_image_path(p) for p in image_paths])
df.head()

In [ ]:
# Count the number of images and number of patients
print("Images:", df["image_path"].nunique())
print("Patients:", df["subject_id"].nunique())

In [ ]:
# Check that all images and masks exist
assert df["image_path"].apply(lambda p: p.exists()).all()
assert df["mask_path"].apply(lambda p: p.exists()).all()

In [ ]:
# Check the number of channels, dimensions, and min/max values of the images
channels = []
heights = []
widths = []
min_value = -np.inf
max_value = np.inf
for path in df["image_path"]:
    image = io.imread(path)
    channels.append(image.shape[2] if image.ndim == 3 else 1)
    heights.append(image.shape[0])
    widths.append(image.shape[1])
    min_value = max(min_value, image.min())
    max_value = min(max_value, image.max())

print("Channels:", set(channels))
print("Heights: min = {:.2f}, mean = {:.2f}, max = {:.2f}".format(min(heights), np.mean(heights), max(heights)))
print("Widths: min = {:.2f}, mean = {:.2f}, max = {:.2f}".format(min(widths), np.mean(widths), max(widths)))
print("Min value:", min_value)
print("Max value:", max_value)

In [ ]:
# Check the number of channels and pixel values of the masks
mask_channels = set()
mask_values = set()
for path in df["mask_path"]:
    mask = io.imread(path)
    mask_channels.add(mask.shape[2 if mask.ndim == 3 else 1])
    mask_values.update(np.unique(mask))

print("Mask channels:", mask_channels)
print("Mask values:", mask_values)

In [ ]:
# Count the number of images per muscle and echogenicity
df.groupby(["muscle", "echogenicity"]).size()

In [ ]:
# Count the number of patients per muscle and echogenicity
df.groupby(["muscle", "echogenicity"])["subject_id"].nunique()

In [ ]:
# Visualize some examples of each muscle and echogenicity (row: echogenicity,
# column: muscle)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i in range(df["echogenicity"].nunique()):
    for j in range(df["muscle"].nunique()):
        example_image_path = df[
            (df["echogenicity"] == df["echogenicity"].unique()[i])
            & (df["muscle"] == df["muscle"].unique()[j])
        ]["image_path"].iloc[0]
        example_image = plt.imread(example_image_path)
        axes[i, j].imshow(example_image, cmap="gray")
        axes[i, j].set_title(
            f"{df['echogenicity'].unique()[i]} {df['muscle'].unique()[j]}"
        )
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize some examples of each muscle and echogenicity (row: echogenicity,
# column: muscle) with masks overlayed
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i in range(df["echogenicity"].nunique()):
    for j in range(df["muscle"].nunique()):
        example_image_path = df[
            (df["echogenicity"] == df["echogenicity"].unique()[i])
            & (df["muscle"] == df["muscle"].unique()[j])
        ]["image_path"].iloc[0]
        example_mask_path = df[
            (df["echogenicity"] == df["echogenicity"].unique()[i])
            & (df["muscle"] == df["muscle"].unique()[j])
        ]["mask_path"].iloc[0]
        example_image = plt.imread(example_image_path)
        example_mask = plt.imread(example_mask_path)
        axes[i, j].imshow(example_image, cmap="gray")
        axes[i, j].imshow(example_mask, alpha=0.5)
        axes[i, j].set_title(
            f"{df['echogenicity'].unique()[i]} {df['muscle'].unique()[j]}"
        )
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()